# 10 — Consolidated evaluation and limitations

Consolidate existing component and AL results, then compute diagnostics from
saved predictions. This notebook is not a replacement for the upstream
experiments. Keep estimator specifications, evaluation populations and metrics
explicit when comparing numbers. Subsampling tables require a verified manifest.


## 0. Configuration, plotting style and result loading


In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
warnings.filterwarnings('ignore')


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import config
from src.models import evaluation

PROC = config.DIR_PROCESSED
FIG  = config.FIGURE_DIR


plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.7,
    'axes.axisbelow': True, 'legend.frameon': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})


C = {
    'sistema':   '#1f5f8b',  
    'novedad':   '#2a9d8f',  
    'aad':       '#55a868',  
    'hibrido':   '#8172b3',  
    'azar':      '#8d8d8d',  
    'acento':    '#dd8452',  
    'malo':      '#c44e52',  
}
CMAP_CM = 'Blues'


CLASS_NAMES = {6:'uLens',15:'TDE',16:'EB',42:'SNII',52:'SNIax',53:'Mira',
                 62:'SNIbc',64:'KN',65:'M-dwarf',67:'SNIa-91bg',88:'AGN',
                 90:'SNIa',92:'RRL',95:'SLSN-I'}
RARE_CLASSES = ['KN','Mira','uLens','SNIa-91bg','SNIax']  

def save(fig, item_name):
    'Save.'
    fig.savefig(FIG / f"10_{item_name}.png", dpi=300, bbox_inches='tight')

print('Proyecto:', PROJECT_ROOT.name)
print('Artefactos in:', PROC)


### Result sources

Read the canonical CSV outputs. Per-object predictions support external
calibration and novelty evaluation. The variant comparison uses holdout
results; the operational internal reference comes from calibrated OOF predictions.


In [ ]:

clf_train_cv = pd.read_csv(PROC / 'resultados_clasificacion_150.csv')   
clf_test = pd.read_csv(PROC / 'resultados_test_batch4.csv')  
pred_b4 = pd.read_csv(PROC / 'predicciones_test_batch4.csv')            


_defs = lambda d: d[~d.variante.str.contains('podad', case=False)].reset_index(drop=True)
clf_train_cv = _defs(clf_train_cv)


from src.models.classification import CLASS_NAMES
clases_14 = sorted(CLASS_NAMES)
metadatos_dist = {
    'train': (config.METADATA_CLEAN_CSV, 'target'),
    'batch1': (config.BatchConfig(1).test_metadata_clean_csv, 'true_target'),
    'batch4': (config.BatchConfig(4).test_metadata_clean_csv, 'true_target'),
}
class_counts = {item_name: pd.read_csv(file_path)[col].value_counts().reindex(clases_14, fill_value=0)
           for item_name, (file_path, col) in metadatos_dist.items()}
distrib_n = pd.DataFrame(class_counts).astype(int)
distrib_n.index = [CLASS_NAMES[c] for c in distrib_n.index]
distrib_n = distrib_n.sort_values('train', ascending=False)
distrib = distrib_n.join((100 * distrib_n / distrib_n.sum()).round(1), rsuffix='_%')


anom_global = pd.read_csv(PROC / 'anomalias_descriptores_batch4.csv')
anom_clase  = pd.read_csv(PROC / 'anomalias_complementariedad_pool.csv')   
anom_consist = pd.read_csv(PROC / 'anomalias_consistencia_batches.csv')   
anom_fusion  = pd.read_csv(PROC / 'anomalias_fusion_reranker.csv')         

N_ANOM_B4 = int(pred_b4.es_desconocida.sum())     
N_TEST_B4 = len(pred_b4)
print(f"Batch 4: {N_TEST_B4:,} objects · {N_ANOM_B4} anomalies reales (class 99) · "
      f"{N_TEST_B4-N_ANOM_B4:,} of classes known")
print('Artefactos loaded correctamente.')


## 1. Objectives, populations and leakage controls

Classification concerns the fourteen known classes. Discovery concerns 99x
objects outside that initial taxonomy. Hold out known objects for evaluation
and share the same seed-specific partitions between policies. Evaluation objects
never enter the AL training set. Simulate a perfect, immediate, uniform-cost
oracle; full light curves are available, so this is not early classification.


In [ ]:

tab_distrib = distrib[['train', 'batch1', 'batch4', 'train_%', 'batch4_%']].copy()
tab_distrib.columns = ['Train (N)', 'Batch 1·DDF (N)', 'Batch 4·WFD (N)', 'Train (%)', 'Batch 4 (%)']
display(tab_distrib.style
        .format({'Train (N)':'{:,.0f}','Batch 1·DDF (N)':'{:,.0f}','Batch 4·WFD (N)':'{:,.0f}',
                 'Train (%)':'{:.1f}','Batch 4 (%)':'{:.1f}'})
        .background_gradient(cmap='Blues', subset=['Train (%)','Batch 4 (%)'])
        .set_caption('Class composition of the three datasets. Training and WFD class prevalence differ.'))


In [ ]:

sort_order = distrib['train_%'].sort_values(ascending=False).index
x = np.arange(len(sort_order)); w = 0.4
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.bar(x - w/2, distrib.loc[sort_order, 'train_%'],  w, label='Train',        color=C['sistema'])
ax.bar(x + w/2, distrib.loc[sort_order, 'batch4_%'], w, label='Batch 4 · WFD', color=C['acento'])
ax.set_xticks(x); ax.set_xticklabels(sort_order, rotation=45, ha='right')
ax.set_ylabel('Object fraction'); ax.yaxis.set_major_formatter(PercentFormatter())
ax.set_title('Class-distribution shift between training and batch 4')
ax.legend(loc='upper right')
plt.tight_layout(); save(fig, 'distribucion_conjuntos'); plt.show()


### Metrics

Macro-F1 weights classes equally; weighted F1 and accuracy reflect the class
mixture. ROC-AUC and precision at K characterize anomaly ranking. Discovery
recall is the number of queried 99x objects divided by all 99x objects in the
query pool. ECE describes top-label confidence calibration; distinguish
multiclass Brier score from the binary top-label diagnostic.

Means, standard deviations and standard errors are different summaries.
Report which one accompanies each figure or table.


## 2. Representation comparison and domain shift


In [ ]:

maestra = clf_train_cv.sort_values('MF1', ascending=False).reset_index(drop=True)
maestra_view = maestra[['variante', 'accuracy', 'WF1', 'MF1']].copy()
display(maestra_view.style
        .format({'accuracy':'{:.3f}','WF1':'{:.3f}','MF1':'{:.3f}'})
        .background_gradient(cmap='Greens', subset=['MF1'])
        .set_caption('Representation variants evaluated by holdout and ordered by macro-F1. This is not the calibrated OOF comparison.'))


In [ ]:

from sklearn.metrics import f1_score
from src.models import classification
tr_oof = classification.features_e2gp('train')
z_oof = np.load(PROC / 'calibracion_e2gp_oof_150_cv3.npz')
pred_oof = z_oof['clases'][z_oof['proba_cal'].argmax(axis=1)]
if len(pred_oof) != len(tr_oof):
    raise ValueError('The OOF cache does not match the current training sample')
mf1_train = f1_score(tr_oof.target, pred_oof, average='macro', zero_division=0)
mf1_test = float(clf_test.iloc[0]['MF1'])
auc_nov = float(clf_test.iloc[0]['auc_novelty_99'])
fig, ax = plt.subplots(figsize=(5.4, 4.2))
barras = ax.bar(['Train\n(OOF calibrated)', 'Batch 4\n(known)'], [mf1_train, mf1_test],
                color=[C['sistema'], C['acento']], width=0.6)
for b, value in zip(barras, [mf1_train, mf1_test]):
    ax.text(b.get_x()+b.get_width()/2, value+.01, f'{value:.3f}', ha='center')
ax.set(ylim=(0,.9), ylabel='Macro-F1', title='Operational E2-GP: internal and external validation')
plt.tight_layout(); save(fig, 'train_vs_test_mf1'); plt.show()
print(f'MF1 OOF = {mf1_train:.4f}; batch 4 = {mf1_test:.4f}; difference = {mf1_test-mf1_train:+.4f}')


import hashlib, json
ruta_fig = config.FIGURE_DIR / '10_train_vs_test_mf1.png'
ruta_fig.with_suffix('.png.meta.json').write_text(json.dumps({
    'semantics': 'oof_calibrated_vs_batch4',
    'figure_sha256': hashlib.sha256(ruta_fig.read_bytes()).hexdigest()
}, indent=2) + '\n', encoding='utf-8')


### Internal and external performance

The calibrated operational classifier has aggregated OOF macro-F1 0.7612 and
batch-4 known-class macro-F1 0.5994, a difference of −0.1618. Do not use the
0.8067 holdout result of another specification as calibrated OOF performance.
Several distributions change simultaneously, so this comparison does not
identify the causal contribution of cadence, noise or class prevalence alone.


## 3. Class-level performance


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

con = pred_b4[~pred_b4.es_desconocida].copy()        
y_true = con.clase_real.values
y_pred = con.pred.values
class_codes = sorted(np.unique(y_true))
display_names = [CLASS_NAMES.get(c, str(c)) for c in class_codes]

rep = classification_report(y_true, y_pred, labels=class_codes, target_names=display_names,
                            output_dict=True, zero_division=0)
tab_clase = (pd.DataFrame(rep).T.loc[display_names, ['precision', 'recall', 'f1-score', 'support']]
             .rename(columns={'precision':'Precisión','recall':'Recall','f1-score':'F1','support':'N'}))
tab_clase['N'] = tab_clase['N'].astype(int)
tab_clase = tab_clase.sort_values('F1', ascending=False)

def resalta_raras(row):
    return ['background-color: #fff3e0' if row.name in RARE_CLASSES else '' for _ in row]

display(tab_clase.style
        .format({'Precisión':'{:.3f}','Recall':'{:.3f}','F1':'{:.3f}','N':'{:,d}'})
        .apply(resalta_raras, axis=1)
        .background_gradient(cmap='RdYlGn', subset=['F1'], vmin=0, vmax=1)
        .set_caption('Informe by class sobre the batch 4 (model definitivo E2-GP). '
                     'En naranja, the 5 classes raras by soporte in the train.'))
print(f"macro-F1 (recalculado by object) = {f1_score(y_true, y_pred, average='macro'):.4f}")


In [ ]:

cm = confusion_matrix(y_true, y_pred, labels=class_codes, normalize='true')
fig, ax = plt.subplots(figsize=(8.4, 7.2))
im = ax.imshow(cm, cmap=CMAP_CM, vmin=0, vmax=1)
ax.set_xticks(range(len(class_codes))); ax.set_xticklabels(display_names, rotation=45, ha='right')
ax.set_yticks(range(len(class_codes))); ax.set_yticklabels(display_names)
ax.set_xlabel('Predicted class'); ax.set_ylabel('True class')
ax.set_title('Confusion matrix — batch 4 (row-normalized)')
for i in range(len(class_codes)):
    for j in range(len(class_codes)):
        v = cm[i, j]
        if v >= 0.01:
            ax.text(j, i, f"{v:.2f}", ha='center', va='center', fontsize=7,
                    color='white' if v > 0.5 else '0.2')
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.set_label('Recall (row fraction)')
plt.tight_layout(); save(fig, 'confusion_batch4'); plt.show()


### Class-level interpretation

Read precision, recall and F1 together with support. Closely related supernova
subtypes remain difficult, and estimates for the smallest classes are unstable.
Aggregate scores cannot replace inspection of confusion between physical classes.


## 4. Probability calibration

Use saved predictions for top-label reliability on known classes of batch 4.
Compare that result with the separate internal OOF calibration experiment.
Calibration on known classes is not a guarantee of out-of-distribution detection.


In [ ]:

conf = con.proba_max.values
acierto = (con.pred.values == con.clase_real.values).astype(float)

def curva_fiabilidad(conf, acierto, n_bins=10):
    bordes = np.linspace(0, 1, n_bins + 1)
    xs, ys, ns = [], [], []
    ece = 0.0; N = len(conf)
    for b in range(n_bins):
        m = (conf > bordes[b]) & (conf <= bordes[b + 1])
        if m.sum() == 0:
            continue
        cf, ac = conf[m].mean(), acierto[m].mean()
        xs.append(cf); ys.append(ac); ns.append(int(m.sum()))
        ece += (m.sum() / N) * abs(ac - cf)
    return np.array(xs), np.array(ys), np.array(ns), ece

xs, ys, ns, ece = curva_fiabilidad(conf, acierto)
brier = float(np.mean((conf - acierto) ** 2))
acc_global = float(acierto.mean()); conf_media = float(conf.mean())
print(f"ECE (top-label, batch 4)   = {ece:.4f}")
print(f"Brier (top-label, batch 4) = {brier:.4f}")
sesgo = acc_global - conf_media
print(f"Confianza mean = {conf_media:.4f}  |  Accuracy = {acc_global:.4f}  "
      f"(sesgo accuracy−confidence = {sesgo:+.4f} → {'infra' if sesgo>0 else 'sobre'}-confianza)")


In [ ]:

fig, (ax, axh) = plt.subplots(2, 1, figsize=(6.4, 6.6), height_ratios=[3, 1], sharex=True)
ax.plot([0, 1], [0, 1], '--', color='0.6', lw=1.2, label='Perfect calibration')
ax.plot(xs, ys, 'o-', color=C['sistema'], lw=2, ms=6, label='E2-GP on batch 4')
for u, txt, col in [(0.30, 'gate novelty', C['novedad']), (0.60, 'auto-clasif.', C['acento'])]:
    ax.axvline(u, color=col, ls=':', lw=1.5)
    ax.text(u, 0.03, f" {u:.2f}\n {txt}", color=col, fontsize=8, rotation=0, va='bottom')
ax.set_ylabel('Fraction correct (accuracy)')
ax.set_title(f"Test-set reliability diagram\nECE = {ece:.3f} · Brier = {brier:.3f}")
ax.legend(loc='upper left'); ax.set_ylim(0, 1)
axh.bar(xs, ns, width=0.08, color=C['sistema'], alpha=0.5)
axh.set_ylabel('Number obj.'); axh.set_xlabel('Predicted confidence c = max_k p(k|x)')
plt.tight_layout(); save(fig, 'calibracion_batch4'); plt.show()


### Calibration interpretation

OOF ECE improves from 0.2172 to 0.0669 and multiclass Brier score from 0.3363
to 0.2618. External batch-4 top-label ECE is approximately 0.0607, with mean
confidence 0.7190 and accuracy 0.7783. Thresholds 0.30 and 0.60 remain design
choices, and high confidence can still hide unknown-class objects.


## 5. Anomaly descriptors, complementarity and novelty ranking


In [ ]:
ag = anom_global.sort_values('ROC_AUC', ascending=False).reset_index(drop=True)
ag_view = ag.rename(columns={'descriptor':'Descriptor'})
display(ag_view.style
        .format({'ROC_AUC':'{:.3f}','P@50':'{:.3f}','P@126':'{:.3f}','P@300':'{:.3f}'})
        .background_gradient(cmap='Greens', subset=['ROC_AUC'])
        .set_caption('Anomaly descriptors in batch 4 (notebook 05). Read subtype diagnostics alongside aggregate ranking scores.'))


In [ ]:


_dcols = ['IsolationForest', 'residuos_GP', 'novedad_1-conf', 'cambios_mag']
display(anom_consist.style
        .format({c: '{:.3f}' for c in _dcols}, na_rep='')
        .background_gradient(cmap='Greens', subset=['novedad_1-conf'])
        .set_caption('Descriptor ROC-AUC across batches (notebook 05). Compare individual batches and subtype-specific performance.'))


_pcols = [c for c in anom_fusion.columns if c.startswith('P@')]
display(anom_fusion.style
        .format({'ROC_AUC': '{:.4f}', **{c: '{:.3f}' for c in _pcols}}, na_rep='')
        .background_gradient(cmap='Greens', subset=['ROC_AUC'])
        .set_caption('Descriptor fusion on the eleven-batch pool. Compare OOF learned fusion with single-score and fixed-weight baselines.'))


In [ ]:

ac = anom_clase.set_index('anomalia')
descr = ['novedad_1-conf', 'IsolationForest', 'residuos_GP', 'cambios_mag']
descr = [d for d in descr if d in ac.columns]
M = ac[descr].astype(float)
etq_tipo = {991:'991 · uLens-Binary (Galactic)', 992:'992 · ILOT', 993:'993 · CART', 994:'994 · PISN'}
M.index = [etq_tipo.get(i, str(i)) for i in M.index]

fig, ax = plt.subplots(figsize=(7.6, 3.6))
im = ax.imshow(M.values, cmap='viridis', vmin=0.4, vmax=0.9, aspect='auto')
ax.set_xticks(range(len(descr))); ax.set_xticklabels(
    ['Novedad\n(1−conf)', 'Isolation\nForest', 'Residuos\nGP', 'Cambios\nmag'])
ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha='center', va='center', fontsize=9,
                color='white' if v < 0.68 else 'black')
ax.set_title('Anomaly-subtype complementarity · eleven-batch pool')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label('ROC-AUC')
plt.tight_layout(); save(fig, 'descriptores_complementariedad'); plt.show()


### Descriptor interpretation

Classifier novelty is useful in the aggregate; Isolation Forest, amplitude and
GP residuals have subtype-specific strengths. No descriptor dominates every
99x population. Complementarity motivates fusion rather than a universal
interpretation of statistical outliers as astrophysical anomalies.


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

y_anom = pred_b4.es_desconocida.values.astype(int)
score  = pred_b4.novelty.values
fpr, tpr, _ = roc_curve(y_anom, score)
prec, rec, _ = precision_recall_curve(y_anom, score)
auc_roc = roc_auc_score(y_anom, score)
auc_pr  = average_precision_score(y_anom, score)
base = y_anom.mean()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.4))
a1.plot(fpr, tpr, color=C['novedad'], lw=2.2, label=f"Novelty (AUC = {auc_roc:.3f})")
a1.plot([0, 1], [0, 1], '--', color='0.6', lw=1, label='Random (0.5)')
a1.set_xlabel('False-positive rate'); a1.set_ylabel('True-positive rate')
a1.set_title('Novelty-channel ROC · batch 4'); a1.legend(loc='lower right')

a2.plot(rec, prec, color=C['novedad'], lw=2.2, label=f"Novelty (PR-AUC = {auc_pr:.3f})")
a2.axhline(base, ls='--', color='0.6', lw=1, label=f"Prevalence ({base:.4f})")
a2.set_xlabel('Recall (class 99)'); a2.set_ylabel('Precision')
a2.set_title('Novelty precision–recall curve · batch 4')
a2.legend(loc='upper right'); a2.set_yscale('log')
plt.tight_layout(); save(fig, 'novelty_roc_pr'); plt.show()
print(f"ROC-AUC = {auc_roc:.4f} (matches with auc_novelty_99={auc_nov:.4f}) | "
      f"PR-AUC = {auc_pr:.4f} vs prevalencia {base:.4f} → factor ×{auc_pr/base:.1f}")


### Ranking limits

A novelty ROC-AUC around 0.68 indicates some ordering information, not reliable
separation of all unknown classes. Precision near the top of the ranking and
discoveries under a fixed budget are the operational questions for the AL study.


## 6. Policies and controlled AL comparison

Random sampling is the reference floor. Triage queries very low confidence
first, then the intermediate region, while removing high-confidence objects.
Two channels reserve separate classification and discovery quotas after the
same removal rule. Use 1,000 labels, rounds of 100 and ten matched seeds.
The five-seed query-criterion study is exploratory and kept separate.


In [ ]:
def _estado_final(dc_csv, tri_csv, frac=0.6):
    'Estado final.'
    dc = pd.read_csv(PROC / dc_csv); tr = pd.read_csv(PROC / tri_csv)
    fdc = dc.sort_values('iter').groupby(['frac_desc', 'semilla']).tail(1)
    ftr = tr.sort_values('iter').groupby(['modo', 'semilla']).tail(1)
    cols = ['MF1', 'recall_anom', 'n_descubiertas']
    return {'two channels': fdc[fdc['frac_desc'] == frac].set_index('semilla')[cols],
            'triage':      ftr[ftr['modo'] == 'triage'].set_index('semilla')[cols],
            'random':   ftr[ftr['modo'] == 'random'].set_index('semilla')[cols]}

def _tabla(o):
    f = []
    for s in ['two channels', 'triage', 'random']:
        d = o[s]; n = len(d)
        f.append({'sistema': s,
                  'MF1': f"{d['MF1'].mean():.3f} ± {d['MF1'].std()/np.sqrt(n):.3f}",
                  'recall_anom': f"{d['recall_anom'].mean():.4f} ± {d['recall_anom'].std()/np.sqrt(n):.4f}",
                  'descub': round(d['n_descubiertas'].mean(), 1)})
    return pd.DataFrame(f).set_index('sistema')

COL = {'two channels': '#2166ac', 'triage': '#d6604d', 'random': '#9aa0a6'}
print('utilidades listas')


## 7. Performance in the reference batch


In [ ]:
o4 = _estado_final('dos_canales_batch4_10seeds.csv', 'triaje_batch4_10seeds.csv')
print('Batch 4 · 10 seeds · mean ± standard error')
display(_tabla(o4))


### Single-batch interpretation

Both active policies outperform random queries. Triage and two channels have
similar discovery totals in batch 4. An observed similarity is not a formal
equivalence test; assess the other batches before generalizing this result.


## 8. Generalization across individual batches


In [ ]:
gpb = pd.read_csv(PROC / 'generalizacion_por_batch.csv')
n = gpb['semilla'].nunique(); sort_order = ['two channels', 'triage', 'random']
agg = (gpb.groupby(['batch', 'sistema'])
       .agg(MF1=('MF1', 'mean'), MF1_ee=('MF1', lambda x: x.std()/np.sqrt(n)),
            recall=('recall', 'mean'), recall_ee=('recall', lambda x: x.std()/np.sqrt(n))).round(4))
print('MF1 by lote and sistema:');                 display(agg['MF1'].unstack()[sort_order])
print('Recall of anomalies by lote and sistema:'); display(agg['recall'].unstack()[sort_order])

lotes = sorted(gpb['batch'].unique()); bx = np.arange(len(lotes)); w = 0.25
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for a, met, yl in [(ax[0], 'MF1', 'MF1'), (ax[1], 'recall', 'recall of anomalies')]:
    for j, s in enumerate(sort_order):
        y = [agg.loc[(b, s), met] for b in lotes]; e = [agg.loc[(b, s), met + '_ee'] for b in lotes]
        a.bar(bx + (j - 1) * w, y, w, yerr=e, capsize=3, color=COL[s], label=s)
    a.set_xticks(bx); a.set_xticklabels([f'batch {b}' for b in lotes]); a.set_ylabel(yl)
    a.grid(axis='y', alpha=.3); a.legend(fontsize=8)
fig.suptitle('Three systems across independent batches'); fig.tight_layout(); plt.show()


### Cross-batch interpretation

Performance and the benefit of a discovery quota vary across batches 1, 4, 5
and 6. This is internal evidence within PLAsTiCC; additional batches from the
same simulator do not establish validity on real Rubin observations.


## 9. The combined-pool regime


In [ ]:
o99 = _estado_final('dos_canales_batch99_10seeds.csv', 'triaje_batch99_10seeds.csv')
print('Full processed pool (~350k) · 10 seeds · mean ± standard error')
display(_tabla(o99))


sis = ['two channels', 'triage', 'random']
nn  = {s: len(o99[s]) for s in sis}
mf  = {s: o99[s]['MF1'].mean() for s in sis};         mfe = {s: o99[s]['MF1'].std()/np.sqrt(nn[s]) for s in sis}
rc  = {s: o99[s]['recall_anom'].mean() for s in sis}; rce = {s: o99[s]['recall_anom'].std()/np.sqrt(nn[s]) for s in sis}
x = np.arange(3); cols = [COL[s] for s in sis]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))


ax[0].bar(x, [mf[s] for s in sis], 0.62, yerr=[mfe[s] for s in sis], capsize=4, color=cols)
ax[0].axhline(mf['random'], ls=':', color='#444', lw=1.2)
ax[0].text(2.5, mf['random'] + 0.001, 'nivel of the azar', va='bottom', ha='right', fontsize=8, color='#444')
ax[0].set_ylim(0.60, 0.69); ax[0].set_xticks(x); ax[0].set_xticklabels(sis, fontsize=9)
ax[0].set_ylabel('Macro-F1 (expanded axis)'); ax[0].set_title('Classification (MF1)'); ax[0].grid(axis='y', alpha=.3)
for xi, s in zip(x, sis):
    ax[0].text(xi, mf[s] + mfe[s] + 0.003, f'{mf[s]:.3f}', ha='center', fontsize=8, fontweight='bold' if s == 'two channels' else 'normal')


top = max(rc.values()) * 1.4
ax[1].bar(x, [rc[s] for s in sis], 0.62, yerr=[rce[s] for s in sis], capsize=4, color=cols)
ax[1].set_ylim(0, top); ax[1].set_xticks(x); ax[1].set_xticklabels(sis, fontsize=9)
ax[1].set_ylabel('recall of anomalies'); ax[1].set_title('Anomaly discovery (recall)'); ax[1].grid(axis='y', alpha=.3)
for xi, s in zip(x, sis):
    ax[1].text(xi, rc[s] + rce[s] + top*0.03, f"{rc[s]:.3f}\n(×{rc[s]/rc['random']:.1f} random)",
               ha='center', fontsize=8, fontweight='bold' if s == 'two channels' else 'normal')
fig.suptitle('Combined pool of eleven processed batches (~350k) · 10 seeds', fontweight='bold')
fig.tight_layout(); fig.savefig(config.FIGURE_DIR / 'evaluacion_escala_completa.png', dpi=150); plt.show()


tr99 = pd.read_csv(PROC / 'triaje_batch99_10seeds.csv')
qa = tr99[tr99['modo'] == 'triage'].sort_values('iter').groupby('semilla').tail(1)['q_anom'].mean()
pr = (pd.read_csv(PROC / 'predicciones_test_batch99.csv')
      .merge(pd.read_csv(PROC / 'test_batch99_metadata_clean.csv'), on='object_id'))
ng = int((pr['confianza'] < 0.30).sum()); na = int((pr['true_target'] >= 990).sum())
ag = int((pr[pr['confianza'] < 0.30]['true_target'] >= 990).sum())
print(f"\nTriage anomaly-channel queries: {qa:.0f} of 1000 (satura the budget).")
print(f"Objetos with conf<0.30: {ng} · anomalies totales: {na} · anomalies dentro of the gate conf<0.30: {ag}.")


### Large-pool interpretation

At the fixed budget, two channels with discovery allocation 0.6 recover a mean
of 38 anomalies, versus 12.5 for triage and 7 for random queries. The confidence
tail can absorb the triage budget before it reaches the intermediate region.
High-confidence removal remains a structural limitation for both active policies.


## 10. Seed variability and allocation sensitivity


In [ ]:
estadistica_99 = evaluation.paired_comparison_summary(
    o99['two channels'], o99['triage'], ['MF1', 'recall_anom'],
    n_boot=20_000, seed_value=config.RANDOM_STATE)
print('Two channels minus triage · full pool · paired seed comparison:')
display(estadistica_99.style.format({
    'media_propuesto': '{:.4f}', 'media_referencia': '{:.4f}',
    'diferencia_media': '{:+.4f}', 'ic95_inf': '{:+.4f}', 'ic95_sup': '{:+.4f}',
    'diferencia_mediana': '{:+.4f}', 'rango_biserial': '{:.3f}',
    'p_wilcoxon': '{:.4f}', 'p_holm': '{:.4f}'}))

dc = pd.read_csv(PROC / 'dos_canales_batch99_10seeds.csv')
fdc = dc.sort_values('iter').groupby(['frac_desc', 'semilla']).tail(1); nn = fdc['semilla'].nunique()
light_curve = (fdc.groupby('frac_desc')
         .agg(MF1=('MF1', 'mean'), MF1_ee=('MF1', lambda x: x.std()/np.sqrt(nn)),
              recall=('recall_anom', 'mean'), recall_ee=('recall_anom', lambda x: x.std()/np.sqrt(nn))).reset_index())
ref = {'triaje_MF1': o99['triage']['MF1'].mean(), 'azar_MF1': o99['random']['MF1'].mean(),
       'triaje_rec': o99['triage']['recall_anom'].mean(), 'azar_rec': o99['random']['recall_anom'].mean()}
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].errorbar(light_curve['frac_desc'], light_curve['MF1'], yerr=light_curve['MF1_ee'], marker='o', color=COL['two channels'], capsize=3, label='two channels')
ax[0].axhline(ref['triaje_MF1'], ls='--', color=COL['triage'], label='triage'); ax[0].axhline(ref['azar_MF1'], ls=':', color=COL['random'], label='Random')
ax[0].set_xlabel('frac_desc'); ax[0].set_ylabel('MF1'); ax[0].grid(alpha=.3); ax[0].legend(fontsize=8)
ax[1].errorbar(light_curve['frac_desc'], light_curve['recall'], yerr=light_curve['recall_ee'], marker='o', color=COL['two channels'], capsize=3, label='two channels')
ax[1].axhline(ref['triaje_rec'], ls='--', color=COL['triage'], label='triage'); ax[1].axhline(ref['azar_rec'], ls=':', color=COL['random'], label='Random')
ax[1].set_xlabel('frac_desc'); ax[1].set_ylabel('recall of anomalies'); ax[1].grid(alpha=.3); ax[1].legend(fontsize=8)
fig.suptitle('Allocation sensitivity in the combined pool'); fig.tight_layout(); plt.show()


### Paired inference

Discovery improves in all ten seeds and remains significant after Holm
adjustment (p ≈ 0.004). The mean macro-F1 difference is positive in nine seeds,
but its adjusted test is not significant (p ≈ 0.064). Bootstrap intervals and
Wilcoxon tests answer related but different questions. Do not claim proven
classification non-inferiority or equivalence from an insignificant difference.


### 10.2. Computational subsampling

Validate the canonical subsample against its disjoint complement in the full
clean batch 4. Use the same classifier and representation, exploratory KS,
proportion and composition diagnostics, seven nonredundant Holm-adjusted tests,
and 200 samples without replacement. Record feature alignment and provenance.

The cached validation completed in the reference Linux environment on 23 October 2026. Median confidence is 0.743237 in the subsample and 0.746207 in the complement. No nonredundant test rejects at 5% after Holm correction; this is not proof of equivalence. The subsample novelty ROC-AUC is 0.668225, versus 0.693729 for the full batch. See the retained verification record and execution archive for provenance.


In [ ]:
from src.models.subsampling import load_validation
tests_sub, remuestreo_sub, realizaciones_sub, procedencia_sub = load_validation(PROC)
display(tests_sub.round(4))
display(remuestreo_sub.round(4))
print('Continuous-variable summary:', procedencia_sub['continuous_summary'])
print('Rows aligned with the canonical subsample:', procedencia_sub['aligned_feature_rows'])


### Reading the regenerated validation

Continuous summaries in the KS table are **medians**, not means. The table
loader rejects outputs without a matching manifest. Read effect sizes,
adjusted p-values and Monte Carlo positions from the newly generated tables;
do not assume the earlier percentiles or p-values still apply. The empirical
scope is this WFD reference batch, not every possible test batch.


## 11. Related work


In [ ]:
comparacion_literatura = pd.DataFrame([
    {
        'trabajo': 'Boone (2019) · avocado',
        'objetivo': 'Static supervised classification',
        'Data / interaction': 'test PLAsTiCC full; without budget of experto',
        'Result / metric': 'log-loss plano ponderado by class = 0.468',
        'Relation to this thesis': 'Classification context; different task and metric',
    },
    {
        'trabajo': 'Fink AL · Leoni (2022), Möller et to the. (2025)',
        'objetivo': 'AL for mejorar classification and seguimiento',
        'Data / interaction': 'ZTF real; 300 queries simuladas (2022) and 92 espectros (2025)',
        'Result / metric': '2025 deployment: fewer spectra and improved reported metrics',
        'Relation to this thesis': 'precedente of the canal of classification; without budget compartido',
    },
    {
        'trabajo': 'Ishida et to the. (2021) · AAD',
        'objetivo': 'discovery activo of anomalies',
        'Data / interaction': 'PLAsTiCC: 7,223 objects and 145 inspections; also OSC',
        'Result / metric': 'More anomalies than static IF in the reported OSC experiment',
        'Relation to this thesis': 'Methodological precedent; different anomaly definition and protocol',
    },
    {
        'trabajo': 'Lochner and Bassett (2021) · Astronomaly',
        'objetivo': 'Personalized active recommendation',
        'Data / interaction': 'Galaxy Zoo + simulation; first 100 inspections',
        'Result / metric': '~2× objects interesantes in the primeras 100',
        'Relation to this thesis': 'antecedente of framework; data and objetivo distintos',
    },
    {
        'trabajo': 'Este TFM · sistema of two channels',
        'objetivo': 'Classification and discovery under a shared budget',
        'Data / interaction': 'pool PLAsTiCC b99 (352 691); 1 000 queries',
        'Result / metric': f"MF1={mf['two channels']:.4f}; recall 99x={rc['two channels']:.4f}",
        'Relation to this thesis': 'benchmark interno directo frente a triage and azar',
    },
])
display(comparacion_literatura.style.hide(axis='index').set_properties(**{'text-align': 'left'}))


### Comparison scope

Avocado is a reference for static photometric classification; active-learning
studies such as Fink motivate spectroscopic prioritization, and AAD and
Astronomaly motivate expert-guided anomaly discovery. Their datasets, costs,
metrics and anomaly definitions differ, so direct numerical comparisons require
a shared protocol and cannot be inferred from unrelated published scores.


## 12. Evidence and research objectives

The representation ablations assess classification inputs; OOF and external
diagnostics assess confidence; subtype comparisons assess complementarity;
matched AL runs assess the shared-budget policy. The combined pool tests an
additional operating regime, not real-time performance on the full Rubin stream.


## 13. Conclusions, limitations and next steps

The main robust result is improved anomaly discovery under a fixed budget in
the larger simulated pool. Classification is higher on average but its paired
advantage over triage is not conclusive with ten seeds. Absolute discovery
recall remains low, and confidence-based removal hides many anomalies.

The study uses simulated complete light curves, ideal labels and uniform costs.
Future work includes partial curves, realistic delayed follow-up, domain shift,
cost-aware selection and adaptive quota allocation. Interpretability explains
the underlying forests and associations, not exact calibrated novelty or
physical causation. The subsampling diagnostics use the canonical final model and compare the
used subsample with its disjoint full-batch complement.
